# Freight Rate Prediction

## Final Model Training & Prediction

This notebook trains the final Linear Regression model on the complete development dataset and generates:

1. Validation-set predictions for the assessment submission.
2. December 2025 freight-rate predictions for the visualization output.

The final model and preprocessing pipeline are also saved as reusable artifacts.

In [1]:
import numpy as np
import pandas as pd
import joblib

from pathlib import Path
from sklearn.linear_model import LinearRegression


In [2]:
PROJECT_ROOT = Path(r"D:\freight-rate-prediction")

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
DATA_DIR = PROJECT_ROOT / "data"

print("Project root:", PROJECT_ROOT)
print("Artifacts directory exists:", ARTIFACTS_DIR.exists())
print("Data directory exists:", DATA_DIR.exists())

Project root: D:\freight-rate-prediction
Artifacts directory exists: True
Data directory exists: True


In [3]:
preprocessor = joblib.load(
    ARTIFACTS_DIR / "preprocessor.joblib"
)

print("Preprocessor loaded successfully.")
print(
    "Number of processed features:",
    len(preprocessor.get_feature_names_out())
)

Preprocessor loaded successfully.
Number of processed features: 148


In [4]:
df_train_test = pd.read_csv(
    DATA_DIR / "train-test.csv"
)

print("Training data shape:", df_train_test.shape)

Training data shape: (48000, 14)


In [5]:
print(df_train_test.shape)
print(df_train_test.columns.tolist())

(48000, 14)
['load_id', 'pickup', 'delivery', 'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'distance', 'equipment', 'weight', 'date', 'market_index', 'quote_signal', 'posted_rate']


In [6]:
df_train_test["date"] = pd.to_datetime(
    df_train_test["date"]
)

print(df_train_test["date"].dtype)

datetime64[ns]


In [7]:
# ============================================================
# Full Feature Engineering
# Applied to the complete development dataset
# ============================================================

# Date-based features
df_train_test["month"] = df_train_test["date"].dt.month
df_train_test["day_of_week"] = df_train_test["date"].dt.dayofweek
df_train_test["day_of_month"] = df_train_test["date"].dt.day
df_train_test["week_of_year"] = (
    df_train_test["date"].dt.isocalendar().week.astype(int)
)

# Geographic differences
df_train_test["lat_diff"] = (
    df_train_test["delivery_lat"] - df_train_test["pickup_lat"]
)

df_train_test["lon_diff"] = (
    df_train_test["delivery_lon"] - df_train_test["pickup_lon"]
)

# Haversine geographic distance
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 3958.8  # Earth radius in miles

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    return 2 * R * np.arcsin(np.sqrt(a))


df_train_test["geographic_distance"] = haversine_distance(
    df_train_test["pickup_lat"],
    df_train_test["pickup_lon"],
    df_train_test["delivery_lat"],
    df_train_test["delivery_lon"]
)

# Log-transformed route efficiency
df_train_test["log_route_efficiency"] = np.log1p(
    df_train_test["distance"]
    / df_train_test["geographic_distance"]
)

# Clean weight
df_train_test["weight_clean"] = (
    df_train_test["weight"].abs()
)

# Weight per mile
df_train_test["weight_per_mile"] = (
    df_train_test["weight_clean"]
    / df_train_test["distance"]
)

print("Feature engineering completed.")
print("Shape:", df_train_test.shape)

Feature engineering completed.
Shape: (48000, 24)


In [8]:
print(df_train_test.columns.tolist())

['load_id', 'pickup', 'delivery', 'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'distance', 'equipment', 'weight', 'date', 'market_index', 'quote_signal', 'posted_rate', 'month', 'day_of_week', 'day_of_month', 'week_of_year', 'lat_diff', 'lon_diff', 'geographic_distance', 'log_route_efficiency', 'weight_clean', 'weight_per_mile']


In [9]:
X_full = df_train_test.drop(columns=["posted_rate"])
y_full = df_train_test["posted_rate"]

print("X_full:", X_full.shape)
print("y_full:", y_full.shape)

X_full: (48000, 23)
y_full: (48000,)


In [10]:
X_full_processed = preprocessor.fit_transform(X_full)

print("Processed shape:", X_full_processed.shape)

Processed shape: (48000, 148)


In [11]:
final_model = LinearRegression()

final_model.fit(
    X_full_processed,
    y_full
)

print("Final Linear Regression trained successfully.")

Final Linear Regression trained successfully.


In [12]:

joblib.dump(
    preprocessor,
    ARTIFACTS_DIR / "final_preprocessor.joblib"
)

joblib.dump(
    final_model,
    ARTIFACTS_DIR / "final_linear_regression.joblib"
)

print("Final preprocessor saved.")
print("Final Linear Regression model saved.")

Final preprocessor saved.
Final Linear Regression model saved.


In [13]:
validation_df = pd.read_csv(
    PROJECT_ROOT / "data" / "validation.csv"
)

print("Validation shape:", validation_df.shape)
print(validation_df.columns.tolist())

Validation shape: (12000, 13)
['load_id', 'pickup', 'delivery', 'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'distance', 'equipment', 'weight', 'date', 'market_index', 'quote_signal']


In [14]:
validation_df["date"] = pd.to_datetime(
    validation_df["date"]
)

print(validation_df["date"].dtype)

datetime64[ns]


In [15]:
# ============================================================
# Feature Engineering for Validation Data
# ============================================================

# Date-based features
validation_df["month"] = validation_df["date"].dt.month
validation_df["day_of_week"] = validation_df["date"].dt.dayofweek
validation_df["day_of_month"] = validation_df["date"].dt.day
validation_df["week_of_year"] = (
    validation_df["date"].dt.isocalendar().week.astype(int)
)

# Geographic differences
validation_df["lat_diff"] = (
    validation_df["delivery_lat"] - validation_df["pickup_lat"]
)

validation_df["lon_diff"] = (
    validation_df["delivery_lon"] - validation_df["pickup_lon"]
)

# Geographic distance
validation_df["geographic_distance"] = haversine_distance(
    validation_df["pickup_lat"],
    validation_df["pickup_lon"],
    validation_df["delivery_lat"],
    validation_df["delivery_lon"]
)

# Log-transformed route efficiency
validation_df["log_route_efficiency"] = np.log1p(
    validation_df["distance"]
    / validation_df["geographic_distance"]
)

# Clean weight
validation_df["weight_clean"] = (
    validation_df["weight"].abs()
)

# Weight per mile
validation_df["weight_per_mile"] = (
    validation_df["weight_clean"]
    / validation_df["distance"]
)

print("Validation feature engineering completed.")
print("Shape:", validation_df.shape)

Validation feature engineering completed.
Shape: (12000, 23)


In [16]:
X_validation = validation_df.drop(
    columns=["load_id", "date"]
)

X_validation_processed = preprocessor.transform(
    X_validation
)

print("Processed validation shape:", X_validation_processed.shape)

Processed validation shape: (12000, 148)


In [17]:
validation_predictions = final_model.predict(
    X_validation_processed
)

print("Predictions shape:", validation_predictions.shape)
print("First 10 predictions:")
print(validation_predictions[:10])

Predictions shape: (12000,)
First 10 predictions:
[ 942.64253751 4941.21360113 5403.39725648 4258.26722777 1904.84622853
 4569.2447702  5331.39049055 2849.4540302  1097.28589022 1087.7137343 ]


In [18]:
template_df = pd.read_csv(
    PROJECT_ROOT / "data" / "validation-predictions-template.csv"
)

print("Template shape:", template_df.shape)
print(template_df.columns.tolist())
print(template_df.head())

Template shape: (12000, 2)
['load_id', 'predicted_rate']
     load_id  predicted_rate
0  TE-000001             NaN
1  TE-000002             NaN
2  TE-000003             NaN
3  TE-000004             NaN
4  TE-000005             NaN


In [19]:
print(
    template_df["load_id"].equals(
        validation_df["load_id"]
    )
)

True


In [20]:
template_df["predicted_rate"] = validation_predictions

validation_predictions_path = (
    PROJECT_ROOT / "validation_predictions.csv"
)

template_df.to_csv(
    validation_predictions_path,
    index=False
)

print("Validation predictions saved successfully.")
print("Path:", validation_predictions_path)
print("Shape:", template_df.shape)
print(template_df.head())

Validation predictions saved successfully.
Path: D:\freight-rate-prediction\validation_predictions.csv
Shape: (12000, 2)
     load_id  predicted_rate
0  TE-000001      942.642538
1  TE-000002     4941.213601
2  TE-000003     5403.397256
3  TE-000004     4258.267228
4  TE-000005     1904.846229


In [21]:
# ============================================================
# Check raw validation predictions
# ============================================================

print("Type:", type(validation_predictions))
print("Shape:", validation_predictions.shape)

print(
    "Minimum prediction:",
    validation_predictions.min()
)

print(
    "Number of non-positive predictions:",
    (validation_predictions <= 0).sum()
)

print("\nFirst non-positive predictions:")

non_positive = validation_predictions[
    validation_predictions <= 0
]

print(non_positive[:20])

Type: <class 'numpy.ndarray'>
Shape: (12000,)
Minimum prediction: -305.14515484100775
Number of non-positive predictions: 18

First non-positive predictions:
[-196.81641626   -8.70727379  -60.33343216  -61.82631384 -305.14515484
 -167.17757005 -163.38288451  -45.95565716  -65.52453628   -6.22174165
 -130.56457624  -77.99762714   -4.73320095  -26.88239872 -132.9060629
 -184.72800867  -72.38621748 -299.01707147]


In [22]:
# ============================================================
# Ensure validation predictions are valid positive rates
# ============================================================

raw_validation_predictions = validation_predictions.copy()

non_positive_mask = validation_predictions <= 0

print("Non-positive predictions before correction:",
      non_positive_mask.sum())

validation_predictions[non_positive_mask] = 1.0

print("Non-positive predictions after correction:",
      (validation_predictions <= 0).sum())

print("Minimum prediction after correction:",
      validation_predictions.min())

Non-positive predictions before correction: 18
Non-positive predictions after correction: 0
Minimum prediction after correction: 1.0


In [23]:
# ============================================================
# Update final validation_predictions.csv
# ============================================================

validation_output = pd.DataFrame({
    "load_id": validation_df["load_id"],
    "predicted_rate": validation_predictions
})

validation_output.to_csv(
    PROJECT_ROOT / "validation_predictions.csv",
    index=False
)

print("Validation predictions updated successfully.")
print("Shape:", validation_output.shape)
print(validation_output.head())

print(
    "\nNon-positive predictions:",
    (validation_output["predicted_rate"] <= 0).sum()
)

Validation predictions updated successfully.
Shape: (12000, 2)
     load_id  predicted_rate
0  TE-000001      942.642538
1  TE-000002     4941.213601
2  TE-000003     5403.397256
3  TE-000004     4258.267228
4  TE-000005     1904.846229

Non-positive predictions: 0


******************************************************************************************************************************

In [24]:
december_df = pd.read_csv(
    PROJECT_ROOT / "data" / "december-chart-inputs.csv"
)

print("December shape:", december_df.shape)
print(december_df.columns.tolist())
print(december_df.head())

December shape: (31, 7)
['pickup', 'delivery', 'distance', 'equipment', 'weight', 'date', 'predicted_rate']
      pickup    delivery  distance equipment  weight        date  \
0  Lexington  Fort Wayne       360   Dry Van   32000  2025-12-01   
1  Lexington  Fort Wayne       360   Dry Van   32000  2025-12-02   
2  Lexington  Fort Wayne       360   Dry Van   32000  2025-12-03   
3  Lexington  Fort Wayne       360   Dry Van   32000  2025-12-04   
4  Lexington  Fort Wayne       360   Dry Van   32000  2025-12-05   

   predicted_rate  
0      797.475506  
1      797.517024  
2      797.558543  
3      797.600061  
4      797.641580  


In [25]:
december_matches = df_train_test[
    (df_train_test["pickup"] == "Lexington")
    & (df_train_test["delivery"] == "Fort Wayne")
    & (df_train_test["distance"] == 360)
    & (df_train_test["equipment"] == "Dry Van")
    & (df_train_test["weight"] == 32000)
]

print("Matching training rows:", len(december_matches))

print(
    december_matches[
        [
            "pickup",
            "delivery",
            "pickup_lat",
            "pickup_lon",
            "delivery_lat",
            "delivery_lon",
            "distance",
            "equipment",
            "weight",
            "market_index",
            "quote_signal",
            "date",
            "posted_rate",
        ]
    ].head(20)
)

Matching training rows: 0
Empty DataFrame
Columns: [pickup, delivery, pickup_lat, pickup_lon, delivery_lat, delivery_lon, distance, equipment, weight, market_index, quote_signal, date, posted_rate]
Index: []


In [26]:
print("Lexington pickup count:",
      (df_train_test["pickup"] == "Lexington").sum())

print("Fort Wayne delivery count:",
      (df_train_test["delivery"] == "Fort Wayne").sum())

Lexington pickup count: 1209
Fort Wayne delivery count: 1176


In [27]:
print("Lexington coordinates:")
print(
    df_train_test.loc[
        df_train_test["pickup"] == "Lexington",
        ["pickup_lat", "pickup_lon"]
    ].drop_duplicates()
)

print("\nFort Wayne coordinates:")
print(
    df_train_test.loc[
        df_train_test["delivery"] == "Fort Wayne",
        ["delivery_lat", "delivery_lon"]
    ].drop_duplicates()
)

Lexington coordinates:
    pickup_lat  pickup_lon
28    36.99152   -84.99876

Fort Wayne coordinates:
    delivery_lat  delivery_lon
24      41.31561     -85.36206


In [28]:
monthly_market = (
    df_train_test
    .groupby(df_train_test["date"].dt.to_period("M"))[
        ["market_index", "quote_signal"]
    ]
    .mean()
)

print(monthly_market)

         market_index  quote_signal
date                               
2025-01      0.926696      2.068436
2025-02      0.995997      2.097130
2025-03      1.072880      2.179539
2025-04      1.207649      1.963388
2025-05      1.301452      1.912829
2025-06      1.280316      2.296007
2025-07      1.200893      1.922997
2025-08      0.981480      2.052200
2025-09      0.893039      2.199785
2025-10      0.957502      1.942758


In [29]:
# ============================================================
# Add geographic coordinates to December data
# ============================================================

december_df["pickup_lat"] = 36.99152
december_df["pickup_lon"] = -84.99876

december_df["delivery_lat"] = 41.31561
december_df["delivery_lon"] = -85.36206

print(december_df.shape)
print(
    december_df[
        [
            "pickup",
            "delivery",
            "pickup_lat",
            "pickup_lon",
            "delivery_lat",
            "delivery_lon"
        ]
    ].head()
)

(31, 11)
      pickup    delivery  pickup_lat  pickup_lon  delivery_lat  delivery_lon
0  Lexington  Fort Wayne    36.99152   -84.99876      41.31561     -85.36206
1  Lexington  Fort Wayne    36.99152   -84.99876      41.31561     -85.36206
2  Lexington  Fort Wayne    36.99152   -84.99876      41.31561     -85.36206
3  Lexington  Fort Wayne    36.99152   -84.99876      41.31561     -85.36206
4  Lexington  Fort Wayne    36.99152   -84.99876      41.31561     -85.36206


In [30]:
# ============================================================
# Geographic feature engineering for December
# ============================================================

december_df["lat_diff"] = (
    december_df["delivery_lat"] - december_df["pickup_lat"]
)

december_df["lon_diff"] = (
    december_df["delivery_lon"] - december_df["pickup_lon"]
)

december_df["geographic_distance"] = haversine_distance(
    december_df["pickup_lat"],
    december_df["pickup_lon"],
    december_df["delivery_lat"],
    december_df["delivery_lon"]
)

# Log-transformed route efficiency
december_df["log_route_efficiency"] = np.log1p(
    december_df["distance"] / december_df["geographic_distance"]
)

print(
    december_df[
        [
            "distance",
            "geographic_distance",
            "log_route_efficiency",
            "lat_diff",
            "lon_diff"
        ]
    ].head()
)

   distance  geographic_distance  log_route_efficiency  lat_diff  lon_diff
0       360           299.401618              0.789547   4.32409   -0.3633
1       360           299.401618              0.789547   4.32409   -0.3633
2       360           299.401618              0.789547   4.32409   -0.3633
3       360           299.401618              0.789547   4.32409   -0.3633
4       360           299.401618              0.789547   4.32409   -0.3633


In [31]:
# ============================================================
# Temporal feature engineering for December
# ============================================================

december_df["date"] = pd.to_datetime(december_df["date"])

december_df["month"] = december_df["date"].dt.month

december_df["day_of_week"] = december_df["date"].dt.dayofweek

december_df["day_of_month"] = december_df["date"].dt.day

december_df["week_of_year"] = (
    december_df["date"]
    .dt.isocalendar()
    .week
    .astype(int)
)

print(
    december_df[
        [
            "date",
            "month",
            "day_of_week",
            "day_of_month",
            "week_of_year"
        ]
    ].head(10)
)

        date  month  day_of_week  day_of_month  week_of_year
0 2025-12-01     12            0             1            49
1 2025-12-02     12            1             2            49
2 2025-12-03     12            2             3            49
3 2025-12-04     12            3             4            49
4 2025-12-05     12            4             5            49
5 2025-12-06     12            5             6            49
6 2025-12-07     12            6             7            49
7 2025-12-08     12            0             8            50
8 2025-12-09     12            1             9            50
9 2025-12-10     12            2            10            50


In [32]:
# ============================================================
# Weight-based features for December
# ============================================================

december_df["weight_clean"] = december_df["weight"].abs()

december_df["weight_per_mile"] = (
    december_df["weight_clean"] / december_df["distance"]
)

print(
    december_df[
        [
            "weight",
            "distance",
            "weight_clean",
            "weight_per_mile"
        ]
    ].head()
)

   weight  distance  weight_clean  weight_per_mile
0   32000       360         32000        88.888889
1   32000       360         32000        88.888889
2   32000       360         32000        88.888889
3   32000       360         32000        88.888889
4   32000       360         32000        88.888889


In [33]:
# ============================================================
# Last available market information before December
# ============================================================

last_market_info = (
    df_train_test
    .sort_values("date")
    .tail(20)[
        ["date", "market_index", "quote_signal"]
    ]
)

print(last_market_info.to_string(index=False))

      date  market_index  quote_signal
2025-10-31       1.01939       2.10755
2025-10-31       0.95720       1.60182
2025-10-31       0.99169       2.08573
2025-10-31       0.96289       1.78149
2025-10-31       0.99026       1.93247
2025-10-31       0.98238       2.07837
2025-10-31       1.00390       2.05376
2025-10-31       1.01361       1.59011
2025-10-31       0.93525       2.18568
2025-10-31       0.95949       2.02498
2025-10-31       0.99072       1.90221
2025-10-31       0.97378       1.99697
2025-10-31       1.01659       2.22890
2025-10-31       1.03082       2.16248
2025-10-31       0.98854       1.99663
2025-10-31       0.99080       1.96459
2025-10-31       0.96171       1.35623
2025-10-31       0.98621       1.76458
2025-10-31       0.99746       2.24777
2025-10-31       1.07073       2.31578


In [34]:
# ============================================================
# Market features for December
# Using the mean of the last available training month (October)
# ============================================================

october_data = df_train_test[
    df_train_test["date"].dt.month == 10
]

december_df["market_index"] = october_data["market_index"].mean()
december_df["quote_signal"] = october_data["quote_signal"].mean()

print(
    december_df[
        ["date", "market_index", "quote_signal"]
    ].head()
)

print("\nDecember market_index:",
      december_df["market_index"].iloc[0])

print("December quote_signal:",
      december_df["quote_signal"].iloc[0])

        date  market_index  quote_signal
0 2025-12-01      0.957502      1.942758
1 2025-12-02      0.957502      1.942758
2 2025-12-03      0.957502      1.942758
3 2025-12-04      0.957502      1.942758
4 2025-12-05      0.957502      1.942758

December market_index: 0.9575017418416129
December quote_signal: 1.942758493715228


In [35]:
# ============================================================
# Final feature lists used by the trained model
# ============================================================

categorical_features = [
    "pickup",
    "delivery",
    "equipment"
]

weight_features = [
    "weight_clean"
]

other_numeric_features = [
    "pickup_lat",
    "pickup_lon",
    "delivery_lat",
    "delivery_lon",
    "distance",
    "market_index",
    "quote_signal",
    "month",
    "day_of_week",
    "day_of_month",
    "week_of_year",
    "lat_diff",
    "lon_diff",
    "geographic_distance",
    "log_route_efficiency",
    "weight_per_mile"
]

december_features = (
    categorical_features
    + weight_features
    + other_numeric_features
)

missing_features = [
    feature
    for feature in december_features
    if feature not in december_df.columns
]

print("Expected features:", len(december_features))
print("Missing features:", missing_features)

print("\nDecember feature matrix shape:")
print(december_df[december_features].shape)

Expected features: 20
Missing features: []

December feature matrix shape:
(31, 20)


In [36]:
print(type(preprocessor))

<class 'sklearn.compose._column_transformer.ColumnTransformer'>


In [37]:
# ============================================================
# Transform December features using the fitted preprocessor
# ============================================================

X_december = december_df[december_features]

X_december_processed = preprocessor.transform(X_december)

print("Raw December shape:", X_december.shape)
print("Processed December shape:", X_december_processed.shape)

Raw December shape: (31, 20)
Processed December shape: (31, 148)


In [38]:
# ============================================================
# Generate December predictions
# ============================================================

december_predictions = final_model.predict(
    X_december_processed
)

print("Predictions shape:", december_predictions.shape)
print("First 10 predictions:")
print(december_predictions[:10])

print("\nMin prediction:", december_predictions.min())
print("Max prediction:", december_predictions.max())

Predictions shape: (31,)
First 10 predictions:
[797.47550556 797.51702411 797.55854265 797.6000612  797.64157975
 797.6830983  797.72461685 809.18095993 809.22247848 809.26399703]

Min prediction: 750.5756111862524
Max prediction: 832.8409799873327


In [39]:
# ============================================================
# Add December predictions to the original December dataframe
# ============================================================

december_df["predicted_rate"] = december_predictions

print(
    december_df[
        [
            "date",
            "pickup",
            "delivery",
            "distance",
            "equipment",
            "weight",
            "predicted_rate"
        ]
    ].head(10)
)

        date     pickup    delivery  distance equipment  weight  \
0 2025-12-01  Lexington  Fort Wayne       360   Dry Van   32000   
1 2025-12-02  Lexington  Fort Wayne       360   Dry Van   32000   
2 2025-12-03  Lexington  Fort Wayne       360   Dry Van   32000   
3 2025-12-04  Lexington  Fort Wayne       360   Dry Van   32000   
4 2025-12-05  Lexington  Fort Wayne       360   Dry Van   32000   
5 2025-12-06  Lexington  Fort Wayne       360   Dry Van   32000   
6 2025-12-07  Lexington  Fort Wayne       360   Dry Van   32000   
7 2025-12-08  Lexington  Fort Wayne       360   Dry Van   32000   
8 2025-12-09  Lexington  Fort Wayne       360   Dry Van   32000   
9 2025-12-10  Lexington  Fort Wayne       360   Dry Van   32000   

   predicted_rate  
0      797.475506  
1      797.517024  
2      797.558543  
3      797.600061  
4      797.641580  
5      797.683098  
6      797.724617  
7      809.180960  
8      809.222478  
9      809.263997  


In [40]:
# ============================================================
# Save December predictions to the original input file
# ============================================================

december_output = december_df[
    [
        "pickup",
        "delivery",
        "distance",
        "equipment",
        "weight",
        "date",
        "predicted_rate"
    ]
].copy()

december_output.to_csv(
    DATA_DIR / "december-chart-inputs.csv",
    index=False
)

print("December predictions saved successfully.")
print("Shape:", december_output.shape)
print("Columns:", list(december_output.columns))
print(december_output.head())

December predictions saved successfully.
Shape: (31, 7)
Columns: ['pickup', 'delivery', 'distance', 'equipment', 'weight', 'date', 'predicted_rate']
      pickup    delivery  distance equipment  weight       date  \
0  Lexington  Fort Wayne       360   Dry Van   32000 2025-12-01   
1  Lexington  Fort Wayne       360   Dry Van   32000 2025-12-02   
2  Lexington  Fort Wayne       360   Dry Van   32000 2025-12-03   
3  Lexington  Fort Wayne       360   Dry Van   32000 2025-12-04   
4  Lexington  Fort Wayne       360   Dry Van   32000 2025-12-05   

   predicted_rate  
0      797.475506  
1      797.517024  
2      797.558543  
3      797.600061  
4      797.641580  
